In [2]:
import pandas

In [9]:
!pip install seaborn
!pip install matplotlib
!pip install scikit-learn

   ---------------------------------------- 0.0/8.9 MB ? eta -:--:--
   -- ------------------------------------- 0.5/8.9 MB 2.4 MB/s eta 0:00:04
   ----- ---------------------------------- 1.3/8.9 MB 4.2 MB/s eta 0:00:02
   -------- ------------------------------- 1.8/8.9 MB 3.1 MB/s eta 0:00:03
   --------- ------------------------------ 2.1/8.9 MB 3.1 MB/s eta 0:00:03
   ---------- ----------------------------- 2.4/8.9 MB 2.2 MB/s eta 0:00:03
   ------------ --------------------------- 2.9/8.9 MB 2.2 MB/s eta 0:00:03
   --------------- ------------------------ 3.4/8.9 MB 2.3 MB/s eta 0:00:03
   ----------------- ---------------------- 3.9/8.9 MB 2.3 MB/s eta 0:00:03
   --------------------- ------------------ 4.7/8.9 MB 2.5 MB/s eta 0:00:02
   --------------------------- ------------ 6.0/8.9 MB 2.8 MB/s eta 0:00:02
   ---------------------------------- ----- 7.6/8.9 MB 3.3 MB/s eta 0:00:01
   ---------------------------------------- 8.9/8.9 MB 3.6 MB/s  0:00:02
   -------------------

In [10]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
from sklearn.model_selection import KFold, cross_val_score
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.metrics import make_scorer, f1_score
import joblib
from sklearn.pipeline import Pipeline

In [11]:
data_path = "aggregated_reviews.csv"
df = pd.read_csv(data_path)
df.head(10)

,domain,review_text,label,label_encoded
0,apparel,GOOD LOOKING KICKS IF YOUR KICKIN IT OLD SCHOO...,positive,1
1,apparel,These sunglasses are all right. They were a li...,positive,1
2,apparel,I don't see the difference between these bodys...,positive,1
3,apparel,Very nice basic clothing. I think the size is...,positive,1
4,apparel,I love these socks. They fit great (my 15 mont...,positive,1
5,apparel,Finally I have found a quality brand of swimsu...,positive,1
6,apparel,Your company was a pleasure to work with- than...,positive,1
7,apparel,very portable. great picture. easy to operate....,positive,1
8,apparel,I have been looking for a pair of Docs for a w...,positive,1
9,apparel,The quality is much better than expected. I bo...,positive,1


In [12]:
def clean_text(text):
    text = re.sub(r'<.*?>', '', str(text))      # Remove HTML tags
    text = re.sub(r'http\S+', '', text)         # Remove URLs
    text = re.sub(r'[^a-zA-Z\s]', '', text)     # Remove special chars & numbers
    text = re.sub(r'\s+', ' ', text).strip()    # Remove extra spaces
    return text.lower()

df['clean_text'] = df['review_text'].apply(clean_text)

df.head()

,domain,review_text,label,label_encoded,clean_text
0,apparel,GOOD LOOKING KICKS IF YOUR KICKIN IT OLD SCHOO...,positive,1,good looking kicks if your kickin it old schoo...
1,apparel,These sunglasses are all right. They were a li...,positive,1,these sunglasses are all right they were a lit...
2,apparel,I don't see the difference between these bodys...,positive,1,i dont see the difference between these bodysu...
3,apparel,Very nice basic clothing. I think the size is...,positive,1,very nice basic clothing i think the size is f...
4,apparel,I love these socks. They fit great (my 15 mont...,positive,1,i love these socks they fit great my month old...


In [13]:
X_train, X_test, y_train, y_test = train_test_split(
    df['clean_text'], df['label_encoded'], 
    test_size=0.2, random_state=42, stratify=df['label_encoded']
)

In [14]:
tfidf = TfidfVectorizer(
    max_features=5000,  # limit vocabulary size
    ngram_range=(1, 2), # include unigrams + bigrams
    stop_words='english'
)

X_tfidf = tfidf.fit_transform(df['clean_text'])
y = df['label_encoded']

In [18]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=200, n_jobs=-1),
    "Linear SVM": LinearSVC(),
    "Random Forest": RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=42)
}


In [16]:
kf = KFold(n_splits=8, shuffle=True, random_state=42)

In [19]:
X = df['clean_text']          # raw cleaned text, NOT the pre-vectorized matrix
y = df['label_encoded']

def make_pipeline(model):
    return Pipeline([
        ('tfidf', TfidfVectorizer(max_features=5000, ngram_range=(1, 2), stop_words='english')),
        ('clf', model)
    ])

cv_results = {}

for name, model in models.items():
    print(f"\n🔹 Evaluating {name} with 8-Fold Cross-Validation (leakage-free)...")
    pipe = make_pipeline(model)
    acc_scores = cross_val_score(pipe, X, y, cv=kf, scoring='accuracy', n_jobs=-1)
    f1_scores  = cross_val_score(pipe, X, y, cv=kf, scoring='f1', n_jobs=-1)

    cv_results[name] = {
        'Accuracy Mean': np.mean(acc_scores),
        'Accuracy Std': np.std(acc_scores),
        'F1 Mean': np.mean(f1_scores),
        'F1 Std': np.std(f1_scores)
    }

    print(f"✅ Accuracy: {np.mean(acc_scores):.4f} ± {np.std(acc_scores):.4f}")
    print(f"✅ F1 Score: {np.mean(f1_scores):.4f} ± {np.std(f1_scores):.4f}")


🔹 Evaluating Logistic Regression with 8-Fold Cross-Validation (leakage-free)...
✅ Accuracy: 0.8572 ± 0.0041
✅ F1 Score: 0.8770 ± 0.0041

🔹 Evaluating Linear SVM with 8-Fold Cross-Validation (leakage-free)...
✅ Accuracy: 0.8516 ± 0.0033
✅ F1 Score: 0.8709 ± 0.0032

🔹 Evaluating Random Forest with 8-Fold Cross-Validation (leakage-free)...
✅ Accuracy: 0.8483 ± 0.0058
✅ F1 Score: 0.8695 ± 0.0046


In [21]:
cv_summary = pd.DataFrame(cv_results).T
cv_summary = cv_summary.sort_values(by='F1 Mean', ascending=False)
print("\n📊 Cross-Validation Results Summary:")
display(cv_summary)



📊 Cross-Validation Results Summary:


,Accuracy Mean,Accuracy Std,F1 Mean,F1 Std
Logistic Regression,0.857188,0.004080,0.876988,0.004140
Linear SVM,0.851636,0.003301,0.870891,0.003210
Random Forest,0.848341,0.005783,0.869546,0.004624


In [ ]:
logistic = cv_summary.index[0]
svm = cv_summary.index[1]
random = cv_summary.index[2]

model_1 = models[logistic]
model_2 = models[svm]
model_3 = models[random]
model_1.fit(X_tfidf, y)
model_2.fit(X_tfidf, y)
model_3.fit(X_tfidf, y)
joblib.dump(model_1, "sentiment_model_lr.pkl")
joblib.dump(model_2, "sentiment_model_svm.pkl")
joblib.dump(model_3, "sentiment_model_rf.pkl")
joblib.dump(tfidf, "tfidf_vectorizer.pkl")
print("✅ Final model and vectorizer saved successfully!")

✅ Final model and vectorizer saved successfully!
